# Exercise 5: gradient boosting with a bag of k-mers

Use the artificial TFBS dataset from Exercise 1 and the same **70% training / 20% validation / 10% test split**.
Keep **k=4 fixed** so we can focus on how boosting works, not on selecting k.

- A **random forest** fits trees independently and averages their probabilities.
- **Gradient boosting** adds shallow trees sequentially. Each tree fits a correction derived from the current model's loss.
- In binary classification, these corrections add to a score (log-odds), which is converted to a probability. They are not votes from independent classifiers.

Watch training and validation **log loss** as trees are added. Choose the number of trees using validation loss, then compare with a decision tree and a random forest at the same k.

In Colab, upload the Exercise 1 CSV using the Files sidebar and set `ifile` below to its filename.
No earlier notebook needs to be running. Stop before the final test cell during discussion: **Run all includes test evaluation**.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, log_loss,
    RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay,
)

ifile = "exercise_1_output/dataset.csv"
seed = 42
k = 4
threshold = 0.5
max_trees = 200
learning_rate = 0.1
max_depth = 2


## 1. Load the artificial data

Each sequence has 100 nucleotides. Label 1 means a motif was planted; label 0 means no motif was planted.


In [ ]:
dataset = pd.read_csv(ifile)

assert dataset["sequence"].str.fullmatch("[ACGT]{100}").all()
assert dataset["label"].isin([0, 1]).all()
assert dataset["label"].nunique() == 2
assert dataset["label"].value_counts().nunique() == 1

print("Sequences:", len(dataset))
print(dataset["label"].value_counts())
dataset.head()


## 2. Keep the same training, validation, and test split

Randomly reserve 10% per label for test and 20% per label for validation, calculated from the original class size.
Use the remaining 70% for training. The test sequences stay unused until the final cell.


In [ ]:
sequences_per_label = len(dataset) // 2
n_test = int(0.10 * sequences_per_label)
n_validation = int(0.20 * sequences_per_label)

test_data = dataset.groupby("label").sample(n=n_test, random_state=seed)
remaining_data = dataset.drop(test_data.index)
validation_data = remaining_data.groupby("label").sample(n=n_validation, random_state=seed)
train_data = remaining_data.drop(validation_data.index)

y_train = train_data["label"]
y_validation = validation_data["label"]

print("Training:", len(train_data))
print("Validation:", len(validation_data))
print("Test (reserved):", len(test_data))


## 3. Count overlapping 4-mers

Move a window one base at a time and count each 4-mer. Each 100-nt sequence contributes 97 windows.
As before, these counts discard position. Fit the vocabulary on training sequences only.
All three models will receive these same sparse count matrices.


In [ ]:
vectorizer = CountVectorizer(
    analyzer="char", ngram_range=(k, k), lowercase=False, dtype=np.int32,
)
x_train = vectorizer.fit_transform(train_data["sequence"])
x_validation = vectorizer.transform(validation_data["sequence"])

print("Training count matrix:", x_train.shape)
print("Validation count matrix:", x_validation.shape)


## 4. Add shallow trees sequentially

The model minimizes **binary log loss**, also called binary cross-entropy: confident wrong predictions receive a large penalty.
Each new tree fits a correction based on the negative gradient of that loss.

- `n_estimators`: the number of boosting rounds (one tree per round here).
- `learning_rate`: scales each tree's contribution. Smaller steps usually need more rounds.
- `max_depth`: limits each tree's depth; here each tree has at most four leaves.

Keep learning rate and depth fixed. Train up to 200 trees, then inspect every intermediate ensemble.
We do not use automatic early stopping: we will choose the tree count explicitly using our existing validation set.


In [ ]:
boosting_model = GradientBoostingClassifier(
    loss="log_loss",
    n_estimators=max_trees,
    learning_rate=learning_rate,
    max_depth=max_depth,
    random_state=seed,
)
boosting_model.fit(x_train, y_train)


## 5. Watch the ensemble learn

`staged_predict_proba` gives predictions after tree 1, then trees 1-2, and so on without fitting again.
Calculate the same log loss on training and validation sequences at every round.

Select the tree count with the **lowest validation log loss**; ties favor fewer trees.
If training loss keeps falling while validation loss rises, that is evidence of overfitting.
The curves may not show overfitting within this range. If the minimum is at the final round, we have reached our budget, not necessarily the best possible tree count.


In [ ]:
training_losses = []
validation_losses = []

for probabilities in boosting_model.staged_predict_proba(x_train):
    training_losses.append(log_loss(y_train, probabilities))

for probabilities in boosting_model.staged_predict_proba(x_validation):
    validation_losses.append(log_loss(y_validation, probabilities))

best_n_trees = int(np.argmin(validation_losses)) + 1
print("Selected number of trees:", best_n_trees)
print("Lowest validation log loss:", round(min(validation_losses), 4))

rounds = range(1, max_trees + 1)
plt.figure(figsize=(9, 5))
plt.plot(rounds, training_losses, label="Training")
plt.plot(rounds, validation_losses, label="Validation")
plt.axvline(best_n_trees, color="black", linestyle="--", label="Selected tree count")
plt.xlabel("Number of trees")
plt.ylabel("Log loss (lower is better)")
plt.title("Gradient boosting at k=4")
plt.legend()
plt.tight_layout()
plt.show()


## 6. Compare three models at k=4

Refit gradient boosting on the **same training sequences**, using the selected tree count.
Do not add validation sequences to training. Also fit the same decision tree and 100-tree random forest used in Exercise 4.

Compare the familiar validation metrics at threshold 0.5. Selection here uses **log loss**, not ROC-AUC, because we are illustrating the loss being optimized.
This is a fixed-setting comparison, not an equally tuned competition between model families.


In [ ]:
models = {
    "Decision tree": DecisionTreeClassifier(random_state=seed),
    "Random forest": RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1),
    "Gradient boosting": GradientBoostingClassifier(
        loss="log_loss", n_estimators=best_n_trees,
        learning_rate=learning_rate, max_depth=max_depth, random_state=seed,
    ),
}

for name, model in models.items():
    model.fit(x_train, y_train)
    print("Fitted:", name)


In [ ]:
validation_probabilities = {}
rows = []
fig, axes = plt.subplots(3, 3, figsize=(17, 15))

for row, (name, model) in enumerate(models.items()):
    probabilities = model.predict_proba(x_validation)[:, 1]
    validation_probabilities[name] = probabilities
    predictions = (probabilities >= threshold).astype(int)
    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_validation, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_validation, predictions),
        "precision": precision_score(y_validation, predictions, zero_division=0),
        "recall": recall_score(y_validation, predictions, zero_division=0),
        "f1": f1_score(y_validation, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_validation, probabilities),
        "average_precision": average_precision_score(y_validation, probabilities),
        "log_loss": log_loss(y_validation, probabilities),
    })
    RocCurveDisplay.from_predictions(y_validation, probabilities, ax=axes[row, 0], name=name)
    axes[row, 0].plot([0, 1], [0, 1], "k--")
    PrecisionRecallDisplay.from_predictions(y_validation, probabilities, ax=axes[row, 1], name=name)
    axes[row, 1].axhline(y_validation.mean(), color="black", linestyle="--")
    ConfusionMatrixDisplay.from_predictions(
        y_validation, predictions, labels=[0, 1], display_labels=["Negative", "Positive"],
        ax=axes[row, 2], colorbar=False, cmap="Blues",
    )
    axes[row, 0].set_title(f"{name}: validation ROC")
    axes[row, 1].set_title(f"{name}: validation precision-recall")
    axes[row, 2].set_title(f"{name}: validation confusion matrix")

validation_results = pd.DataFrame(rows).set_index("model")
print(validation_results.round(4).to_string())
plt.tight_layout()
plt.show()


## 7. Final test evaluation: selected gradient boosting model only

Stop here until the tree count and all other settings are fixed.
Evaluate gradient boosting once on the reserved test set; do not choose settings from these results.
The decision tree and random forest comparison stays on validation data.

**Run all includes this test evaluation.**


In [ ]:
y_test = test_data["label"]
x_test = vectorizer.transform(test_data["sequence"])
test_probabilities = models["Gradient boosting"].predict_proba(x_test)[:, 1]
test_predictions = (test_probabilities >= threshold).astype(int)

test_results = pd.DataFrame([{
    "model": "Gradient boosting",
    "trees": best_n_trees,
    "accuracy": accuracy_score(y_test, test_predictions),
    "balanced_accuracy": balanced_accuracy_score(y_test, test_predictions),
    "precision": precision_score(y_test, test_predictions, zero_division=0),
    "recall": recall_score(y_test, test_predictions, zero_division=0),
    "f1": f1_score(y_test, test_predictions, zero_division=0),
    "roc_auc": roc_auc_score(y_test, test_probabilities),
    "average_precision": average_precision_score(y_test, test_probabilities),
    "log_loss": log_loss(y_test, test_probabilities),
}]).set_index("model")
print("Final test metrics:")
print(test_results.round(4).to_string())

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
RocCurveDisplay.from_predictions(y_test, test_probabilities, ax=axes[0], name="Gradient boosting")
axes[0].plot([0, 1], [0, 1], "k--")
PrecisionRecallDisplay.from_predictions(y_test, test_probabilities, ax=axes[1], name="Gradient boosting")
axes[1].axhline(y_test.mean(), color="black", linestyle="--")
ConfusionMatrixDisplay.from_predictions(
    y_test, test_predictions, labels=[0, 1], display_labels=["Negative", "Positive"],
    ax=axes[2], colorbar=False, cmap="Blues",
)
axes[0].set_title("Gradient boosting: test ROC")
axes[1].set_title("Gradient boosting: test precision-recall")
axes[2].set_title("Gradient boosting: test confusion matrix")
plt.tight_layout()
plt.show()
